<!--
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# Notebook 04: Inference and Embedding Extraction

We extract a **512-dimensional embedding per labelled transaction** from the pretrained
model using **last-token pooling** -- the hidden state at the last non-padding position,
which has attended to the entire input through causal attention.

---

### The input is a history window, not a single row

The TabFormer blueprint encoded each transaction as its own 12-token sequence. That
throws away the only thing pretraining taught the model: how a transaction relates to
what came before it on the same account. The embedding then largely re-encodes the row's
own fields, which XGBoost already has.

Here, each labelled transaction is embedded **in context**:

```
<bos> txn[i-K+1] <sep> txn[i-K+2] <sep> ... <sep> txn[i] <eos> <pad> ...
```

with `K = HISTORY_WINDOW` preceding transactions of the **same sequence**, ending at the
transaction being scored. Under the default `sequence_scope="customer"` that window
spans all of the customer's accounts, interleaved chronologically, with each transaction
carrying its `ACCTH` token; under `"account"` it stays within one account. Padding stays
on the right, so last-token pooling reads the `<eos>` position. Early transactions
simply get a shorter window -- never left-padding, never future transactions (which
would leak).

Two details that matter:

- **History comes from the full split, not from the sample.** Windows are built over
  the complete `{split}.parquet` and only the labelled rows are pushed through the
  model (`HistoryWindowEncoder.select`). Building them from the 100k evaluation subset
  would splice together transactions that were never consecutive.
- **Row alignment is explicit.** `preprocess()` sorts by account and time, so labels
  and raw features are taken from the *sorted* frame at the selected positions and
  saved alongside the embeddings. Notebook 05 reads those files directly instead of
  re-deriving a sample from a random seed.

Set `HISTORY_WINDOW = 1` to reproduce the original per-transaction behaviour.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="torch.cuda")
warnings.filterwarnings("ignore", message="The '.*' attribute with value.*was provided")

import json
import sys
import time
from pathlib import Path

import cudf
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.decoder_inference import HuggingFaceDecoderInference
from src.tokenizer import (
    BankTransactionTokenizer,
    BankTransferTokenizerPipeline,
    HistoryWindowEncoder,
)

from src.experiment import shared_data_dir
DATA_DIR = shared_data_dir()          # set in configs/paths.yaml (the only place)
SPLIT_DIR = DATA_DIR / "temporal_split"
TOKENIZER_DIR = DATA_DIR / "tokenizer"
CORPUS_DIR = DATA_DIR / "decoder_corpus"
EMBED_DIR = DATA_DIR / "embeddings"
MODEL_DIR = DATA_DIR / "model"        # interactive runs; experiments use their run folder
# Run by scripts/run_experiment.py: every path comes from the run folder.
# Interactive use (LTM_EXPERIMENT unset): EXP is None and nothing changes.
from src.experiment import active_experiment
EXP = active_experiment()
if EXP is not None:
    globals().update(EXP.paths())
EMBED_DIR.mkdir(parents=True, exist_ok=True)

STATE_PATH = TOKENIZER_DIR / "tokenizer_state.json"
assert STATE_PATH.exists(), f"Missing {STATE_PATH}  (run notebook 02 first)"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================================================================
# EDIT HERE
# ======================================================================
HISTORY_WINDOW = 32     # transactions per embedding (1 = single transaction)
BATCH_SIZE = 256        # sequences per forward pass; lower if you hit OOM
RESTRICT_TO_SEQUENCES = True   # skip customers/accounts with no labelled row
# "last_token" | "mean" | "last_txn" | "surprise" (anomaly profile), or
# concatenated: "mean+last_txn", "mean+surprise"
# (POOLINGS in src/decoder_inference.py), or "learned" (section 2b). Changing
# it -- or the tokenizer, the checkpoint, the window -- recomputes automatically.
POOLING = "mean"
# Carry each sequence's trailing history across the split boundary.
# Windows used to be built from one split at a time, so a test transaction could
# see no further back than the test cutoff: train rows got 32-deep windows drawn
# from two years of history, test rows near the cutoff got 1-5 transactions and
# padding. That is a train/test mismatch in the model *input*, and removing it is
# not leakage -- reading a customer's own earlier transactions is exactly what a
# production scorer has. Only HISTORY_WINDOW - 1 rows per sequence are needed, so
# the cost is bounded by n_sequences x 31 rows, not by the size of the earlier
# splits. Set False to reproduce the old, truncated behaviour.
HISTORY_CONTEXT = True
# "all": notebook 01's labelled rows. "first_flag": only each customer's first flagged
# transfer counts as fraud, later ones are left out (scripts/first_flag_subsets.py).
LABELS = "all"
FORCE_RECOMPUTE = False   # cached arrays are invalidated automatically below
# Learned pooling only (POOLING containing "learned", section 2b): an attention
# head trained on the label, over the frozen model. It is fitted on a share of
# the TRAIN customers; notebook 05 trains every XGBoost arm on the other ones.
HEAD_FRACTION = 0.5           # share of train customers that fit the head
HEAD_MAX_NEGATIVES = 100_000  # negatives sampled from that share (all positives kept)
HEAD_SEED = 0
# Fine-tuning (with learned pooling only): "none" | "top:N" | "lora:R" | "full"
# (src/finetune.py). Kept only if it beats the head alone on the early-stop slice.
FINETUNE = "none"
FINETUNE_LR = 1e-5        # model weights; the head uses 10x more
FINETUNE_EPOCHS = 3
FINETUNE_BATCH = 32       # windows per step; halved into micro-batches on OOM
if EXP is not None:
    globals().update(EXP.embedding_settings())   # POOLING, HISTORY_WINDOW, ...
# ======================================================================

print(f"Device      : {DEVICE}")
print(f"Tokenizer   : {STATE_PATH}")
print(f"Embeddings  : {EMBED_DIR}")

## 1. Load the Fitted Tokenizer and the Pretrained Model

The tokenizer is **reloaded from the fitted state**, never rebuilt: token IDs have to be
identical to the ones the model was trained on. The assertion on `vocab_size` below is
the guard that catches the classic mistake of pointing this notebook at a checkpoint
trained on a different token space.

In [ ]:
pipeline = BankTransferTokenizerPipeline.from_state(json.loads(STATE_PATH.read_text()))
tokenizer = BankTransactionTokenizer(pipeline)
SCHEMA = pipeline.schema
GROUP_COLS = pipeline.group_cols          # honours config.sequence_scope
TOKENS_PER_TXN = pipeline.tokens_per_transaction
MAX_LENGTH = HistoryWindowEncoder.required_length(HISTORY_WINDOW, TOKENS_PER_TXN)
# An embedding window longer than the sequences the model was pretrained on asks
# it for positions it never saw: refuse rather than embed out of distribution.
_report = CORPUS_DIR / "corpus_report.json"
PRETRAIN_CONTEXT = json.loads(_report.read_text())["context_window"] if _report.exists() else None
assert PRETRAIN_CONTEXT is None or MAX_LENGTH <= PRETRAIN_CONTEXT, (
    f"HISTORY_WINDOW={HISTORY_WINDOW} is {MAX_LENGTH} tokens, longer than the "
    f"{PRETRAIN_CONTEXT}-token sequences the model was pretrained on: lower it, or "
    "pretrain with a longer corpus.context_window"
)

print(f"Tokenizer   : vocab_size={tokenizer.vocab_size}, {TOKENS_PER_TXN} tokens/transaction")
print(f"Window      : {HISTORY_WINDOW} transactions -> {MAX_LENGTH} tokens")
print(f"Sequence    : scope={pipeline.config.sequence_scope}, key={GROUP_COLS}")

# Resolve the checkpoint recorded by notebook 03.
ckpt_info = MODEL_DIR / "latest_checkpoint.json"
if ckpt_info.exists():
    MODEL_PATH = Path(json.loads(ckpt_info.read_text())["checkpoint"])
else:
    # Completed checkpoints only (training_state.json is written last): a save
    # killed mid-write leaves config.json beside truncated weights.
    from src.train_decoder import completed_checkpoints
    _done = completed_checkpoints(MODEL_DIR / "checkpoints")
    found = [_done[step] for step in sorted(_done, reverse=True)]
    assert found, (
        f"No checkpoint under {MODEL_DIR / 'checkpoints'}. Run notebook 03 first."
    )
    MODEL_PATH = found[0]

# Content fingerprints: what the cache key and the guard below compare.
from src.fingerprint import dir_fingerprint, file_fingerprint

TOKENIZER_FP = file_fingerprint(STATE_PATH)
MODEL_FP = dir_fingerprint(MODEL_PATH)
_trained_on = (
    json.loads(ckpt_info.read_text()).get("tokenizer_fingerprint")
    if ckpt_info.exists() else None
)
assert _trained_on in (None, TOKENIZER_FP), (
    "The tokenizer changed since this model was trained (tokenizer_state.json "
    f"is {TOKENIZER_FP}, the model was trained under {_trained_on}). Token IDs no "
    "longer mean what the model learned: re-run notebook 02 (corpus) and 03 "
    "(training) before extracting embeddings."
)

model_cfg = json.loads((MODEL_PATH / "config.json").read_text())
assert model_cfg["vocab_size"] >= tokenizer.vocab_size, (
    f"Checkpoint vocab_size={model_cfg['vocab_size']} < tokenizer vocab_size="
    f"{tokenizer.vocab_size}. This checkpoint was trained on a different token space; "
    "re-run notebook 03 with --model.config.vocab_size from corpus_report.json."
)
print(f"\nCheckpoint  : {MODEL_PATH}")
print(f"  vocab_size={model_cfg['vocab_size']}, hidden={model_cfg.get('hidden_size')}, "
      f"layers={model_cfg.get('num_hidden_layers')}")

inference = HuggingFaceDecoderInference(
    model_path=MODEL_PATH,
    tokenizer=tokenizer,
    pooling=POOLING,
    tokens_per_txn=TOKENS_PER_TXN,  # used by "last_txn"
)
print(f"Model loaded on {inference.device} (embed_dim={inference.embedding_dim})")

## 2. Extract Embeddings

For each split: read the full temporal split, keep the sequences that contain labelled
rows (in customer scope that means every account of those customers), tokenize, build
history windows, then run the model only on the labelled rows.

Cost scales with `HISTORY_WINDOW`: a 32-transaction window is ~480 tokens per row
versus ~16 for a single transaction. If the training split is too large to tokenize in
one pass, lower `NEG_PER_POS` or set `EVAL_SAMPLES` in notebook 01 (fewer accounts survive the
restriction) or lower `HISTORY_WINDOW`.

### 2b. Learned pooling (only when `POOLING` contains `"learned"`)

`mean` and `last_token` pool the window with fixed weights. Attention inside the decoder
mixes the window too, but it was trained to predict the next token. Here a small head
learns, **from the fraud label**, which transactions of the window matter: each
transaction becomes one vector (mean of its field tokens' hidden states), a scorer
weights them (with a learned recency bias), and the weighted sum is the embedding. The
decoder stays frozen; only the head trains.

- **No leakage into XGBoost.** The head is fitted on `HEAD_FRACTION` of the *training
  customers*; notebook 05 trains every arm on the other customers' rows. Early stopping
  uses a customer-disjoint 20% of the head's share -- val and test are never touched.
- **Memory.** Only the head's rows are held as per-transaction states (all positives +
  `HEAD_MAX_NEGATIVES` negatives, ~32 KB per row in fp16).
- **Outputs.** `{split}_embeddings.npy` as usual, plus `{split}_head_scores.npy` (the
  head's own fraud logit) and `attention_head.pt`.

In [ ]:
from src.learned_pooling import (
    collect_states, customer_share, head_scores, save_head, train_head, use_head,
)

HEAD = None
HEAD_PATH = EMBED_DIR / "attention_head.pt"


def fit_learned_head(encoder, labels, processed, sel):
    """Train the head on a share of the train customers; return that share's row mask."""
    global HEAD
    customers = processed[GROUP_COLS[0]].to_pandas().values[sel]
    head_rows = customer_share(customers, HEAD_FRACTION, HEAD_SEED)

    rng = np.random.default_rng(HEAD_SEED)
    pos = np.where(head_rows & (labels == 1))[0]
    neg = np.where(head_rows & (labels == 0))[0]
    neg = rng.choice(neg, min(len(neg), HEAD_MAX_NEGATIVES), replace=False)
    rows = np.sort(np.concatenate([pos, neg]))
    print(f"  Head: {head_rows.mean():.0%} of train rows (by customer) -> "
          f"{len(pos):,} positives + {len(neg):,} negatives")

    head_encoder = encoder.select(encoder.row_index[rows])
    states, valid = collect_states(inference, head_encoder, batch_size=BATCH_SIZE)
    stop = customer_share(customers[rows], 0.2, HEAD_SEED + 1)   # early stopping
    HEAD, hist = train_head(states, valid, labels[rows], stop, device=DEVICE, seed=HEAD_SEED)
    del states, valid

    if FINETUNE != "none":
        # Part of the model trains with the head (src/finetune.py); the model is
        # changed in place, so every embedding below comes from it.
        from src.finetune import finetune_with_head
        HEAD, ft = finetune_with_head(
            inference, head_encoder, labels[rows], stop, HEAD, FINETUNE,
            head_auc=hist["best_stop_auc"], lr=FINETUNE_LR, head_lr=10 * FINETUNE_LR,
            epochs=FINETUNE_EPOCHS, batch_size=FINETUNE_BATCH, seed=HEAD_SEED,
        )
        hist["finetune"] = ft
        if ft["kept"]:
            torch.save(inference._backbone.state_dict(), EMBED_DIR / "finetuned_model.pt")
            print(f"  Fine-tuned model -> {EMBED_DIR / 'finetuned_model.pt'}")

    save_head(HEAD, HEAD_PATH, {"history": hist, "rows": int(len(rows))})
    use_head(HEAD, TOKENS_PER_TXN)          # POOLING "learned" now resolves to it
    print(f"  Head trained: best early-stop AUC {hist['best_stop_auc']:.4f} -> {HEAD_PATH.name}")
    return head_rows


def learned_part(emb):
    """The "learned" slice of a (possibly concatenated) embedding."""
    from src.decoder_inference import pooling_slice
    return emb[:, pooling_slice(POOLING, "learned", inference._embedding_dim_base,
                                TOKENS_PER_TXN)]

In [ ]:
SPLIT_SOURCES = {
    "train": ("train.parquet", "train_balanced.parquet"),
    "val": ("val.parquet", "val_eval.parquet"),
    "test": ("test.parquet", "test_eval.parquet"),
}
if LABELS == "first_flag":
    SPLIT_SOURCES = {s: (full, sub.replace(".parquet", "_first_flag.parquet"))
                     for s, (full, sub) in SPLIT_SOURCES.items()}
    assert all((SPLIT_DIR / sub).exists() for _, sub in SPLIT_SOURCES.values()), (
        "run scripts/first_flag_subsets.py first")

# Splits that precede each split in time, oldest first. Their trailing rows are
# prepended as history-only context; they are never scored. `__split__` is what
# keeps them out of the selection -- `__row_id__` restarts at 0 in every split
# (notebook 01 assigns np.arange per split), so it is NOT unique across a
# concatenation and cannot identify a labelled row on its own.
HISTORY_SOURCES = {"train": [], "val": ["train"], "test": ["train", "val"]}

# Raw columns saved beside the embeddings for notebook 05: the schema fields plus
# any field added through CONFIG.extra_fields, so the raw arm can use them too.
FEATURE_COLS = SCHEMA.feature_columns + [
    spec["column"] for spec in pipeline.config.extra_fields.values()
    if spec["column"] not in SCHEMA.feature_columns
] + ([SCHEMA.description] if SCHEMA.description else []) + (
    ["amt_rel"] if pipeline.config.include_relative_amount else []   # derived, see NB02
)
KEEP_COLS = FEATURE_COLS + ["_target", "__row_id__"]

# A cached .npy is only reusable if it was produced by these settings. cell-07
# rewrites metadata.json unconditionally, so it could name a fresh checkpoint
# while the arrays came from an older one; comparing the *previous* metadata
# before reusing anything closes that. Trust this, not file mtimes.
CACHE_KEY = {
    "model_path": str(MODEL_PATH),
    # Contents, not paths: re-fitting the tokenizer or retraining into the
    # same directory keeps the path and must still invalidate the cache.
    "tokenizer_state": TOKENIZER_FP,
    "model_fingerprint": MODEL_FP,
    "history_window": HISTORY_WINDOW,
    "history_context": HISTORY_CONTEXT,
    "restrict_to_sequences": RESTRICT_TO_SEQUENCES,
    "pooling": POOLING,
    "labels": LABELS,
    "learned_head": (
        {"fraction": HEAD_FRACTION, "max_negatives": HEAD_MAX_NEGATIVES, "seed": HEAD_SEED,
         "finetune": FINETUNE, "finetune_lr": FINETUNE_LR,
         "finetune_epochs": FINETUNE_EPOCHS, "finetune_batch": FINETUNE_BATCH}
        if "learned" in POOLING else None
    ),
    # Re-running notebook 01 with a different NEG_PER_POS / EVAL_SAMPLES rewrites
    # train_balanced.parquet with a different set of labelled rows, which nothing
    # else in this key would notice. Byte size is a free proxy -- a different
    # sample of the same split does not compress to the same length.
    "subset_bytes": {
        subset_name: (SPLIT_DIR / subset_name).stat().st_size
        for _, subset_name in SPLIT_SOURCES.values()
    },
}
_meta_path = EMBED_DIR / "metadata.json"
_prev_meta = json.loads(_meta_path.read_text()) if _meta_path.exists() else {}
_changed = [k for k, v in CACHE_KEY.items() if _prev_meta.get(k) != v]
REUSE_CACHE = not FORCE_RECOMPUTE and bool(_prev_meta) and not _changed
# Learned pooling trains its head during the train pass: if any split must be
# recomputed, recompute them all, or later splits would have no head.
if REUSE_CACHE and "learned" in POOLING and not all(
        (EMBED_DIR / f"{s}_embeddings.npy").exists() for s in SPLIT_SOURCES):
    REUSE_CACHE = False
if _prev_meta and _changed:
    print(f"Cached embeddings were built with different {_changed} -- recomputing.\n")


def seq_key(gdf):
    """Single string key per sequence (for cheap isin filtering)."""
    key = gdf[GROUP_COLS[0]].astype("str")
    for col in GROUP_COLS[1:]:
        key = key + "|" + gdf[col].astype("str")
    return key


def tail_per_sequence(gdf, n):
    """The last *n* transactions of each sequence, chronologically ordered.

    Sorts with time descending, keeps ``cumcount() < n``, then restores ascending
    order. Only groupby().cumcount() is used -- the same primitive the rest of
    this project relies on -- and the frame is grouped by the sort beforehand, so
    the counter is aligned with row order either way.
    """
    if n <= 0:
        return gdf.iloc[:0]
    newest_first = [True] * len(GROUP_COLS) + [False]
    gdf = gdf.sort_values(
        GROUP_COLS + [SCHEMA.datetime], ascending=newest_first
    ).reset_index(drop=True)
    gdf = gdf[gdf.groupby(GROUP_COLS).cumcount() < n]
    return gdf.sort_values(GROUP_COLS + [SCHEMA.datetime]).reset_index(drop=True)


split_sizes = {}

for split, (full_name, subset_name) in SPLIT_SOURCES.items():
    embed_path = EMBED_DIR / f"{split}_embeddings.npy"
    label_path = EMBED_DIR / f"{split}_labels.npy"
    feature_path = EMBED_DIR / f"{split}_features.parquet"

    if embed_path.exists() and REUSE_CACHE:
        emb = np.load(embed_path)
        lbl = np.load(label_path)
        split_sizes[split] = len(emb)
        print(f"[{split}] already extracted: {emb.shape}, "
              f"{int(lbl.sum()):,} positives / {len(lbl):,}")
        continue

    print(f"\n{'=' * 66}\nExtracting {split} embeddings\n{'=' * 66}")
    t0 = time.time()

    subset = cudf.read_parquet(
        str(SPLIT_DIR / subset_name), columns=["__row_id__"] + GROUP_COLS
    )
    full = cudf.read_parquet(str(SPLIT_DIR / full_name))
    full["__split__"] = split
    print(f"  Full split: {len(full):,} rows | labelled subset: {len(subset):,} rows")

    keep = seq_key(subset).unique() if RESTRICT_TO_SEQUENCES else None
    if keep is not None:
        # Keep whole sequences, not just the labelled rows: the history of a
        # labelled transaction is what we are here for. In customer scope this
        # keeps every account of a selected customer.
        full = full[seq_key(full).isin(keep)].reset_index(drop=True)
        print(f"  Restricted to {len(keep):,} sequences -> {len(full):,} rows of history")

    if HISTORY_CONTEXT and HISTORY_SOURCES[split]:
        parts = []
        for source in HISTORY_SOURCES[split]:
            ctx = cudf.read_parquet(str(SPLIT_DIR / f"{source}.parquet"))
            if keep is not None:
                ctx = ctx[seq_key(ctx).isin(keep)].reset_index(drop=True)
            # Capped per source before concatenating: a window never needs more
            # than HISTORY_WINDOW - 1 predecessors, and the earlier splits are
            # millions of rows that would otherwise be preprocessed for nothing.
            ctx = tail_per_sequence(ctx, HISTORY_WINDOW - 1)
            ctx["__split__"] = source
            parts.append(ctx)
            print(f"  Context from {source}: {len(ctx):,} rows "
                  f"(<= {HISTORY_WINDOW - 1} per sequence)")
        # Re-cap after concatenation: the trailing 31 of (train then val) can be
        # drawn from either, and only their union was capped above.
        ctx = tail_per_sequence(cudf.concat(parts, ignore_index=True), HISTORY_WINDOW - 1)
        full = cudf.concat([ctx, full], ignore_index=True)
        del parts, ctx
        print(f"  Frame with context: {len(full):,} rows")

    processed = pipeline.preprocess(full)
    token_df = pipeline.transform(processed)
    print(f"  Tokenized in {time.time() - t0:.1f}s")

    encoder = HistoryWindowEncoder.from_token_frame(
        token_df,
        processed,
        group_cols=GROUP_COLS,
        vocab=tokenizer.vocab,
        unk_id=tokenizer.unk_token_id,
        window=HISTORY_WINDOW,
        max_length=MAX_LENGTH,
        pad_id=tokenizer.pad_token_id,
        bos_id=tokenizer.bos_token_id,
        eos_id=tokenizer.eos_token_id,
        sep_id=tokenizer.sep_token_id,
    )

    # Labelled rows of *this* split, located in the sorted frame by their split
    # row id. The `__split__` guard is required -- see HISTORY_SOURCES above.
    selected_mask = (processed["__split__"] == split) & processed["__row_id__"].isin(
        subset["__row_id__"]
    )
    sel = np.where(selected_mask.to_pandas().values)[0]
    assert len(sel) == len(subset), (
        f"Found {len(sel)} of {len(subset)} labelled rows in the full split -- "
        "were the parquets written by the same notebook 01 run?"
    )
    encoder = encoder.select(sel)
    print(f"  Windows: {len(encoder):,} rows x {encoder.sequence_length} tokens")

    # How much history the windows actually carry. Before HISTORY_CONTEXT this
    # was the tell that val/test rows were being embedded from a truncated past:
    # train rows filled all 32 slots, test rows near the cutoff had 1-5.
    depth = np.minimum(encoder.positions[sel] + 1, HISTORY_WINDOW)
    print(f"  Window depth: mean {depth.mean():.1f} of {HISTORY_WINDOW} txns, "
          f"{(depth == HISTORY_WINDOW).mean():.1%} full")

    labels = processed["_target"].to_pandas().values[sel].astype(np.int8)
    features = processed.take(sel)[KEEP_COLS]

    if "learned" in POOLING and split == "train":
        head_rows = fit_learned_head(encoder, labels, processed, sel)
        np.save(EMBED_DIR / "train_head_rows.npy", head_rows)

    t1 = time.time()
    emb = inference.extract_embeddings_windowed(
        encoder, batch_size=BATCH_SIZE, show_progress=True
    )
    inf_time = time.time() - t1
    print(f"  Embedded {emb.shape} in {inf_time:.1f}s ({len(emb) / max(inf_time, 1e-9):,.0f} rows/s)")

    np.save(embed_path, emb)
    if "learned" in POOLING:
        # The head's own fraud score, straight from its output layer.
        np.save(EMBED_DIR / f"{split}_head_scores.npy", head_scores(HEAD, learned_part(emb)))
    np.save(label_path, labels)
    features.to_parquet(str(feature_path), index=False)
    split_sizes[split] = len(emb)
    print(f"  Saved -> {embed_path.name}, {label_path.name}, {feature_path.name}")

    del full, processed, token_df, encoder
    torch.cuda.empty_cache()

In [ ]:
metadata = {
    "backend": "huggingface_decoder",
    # Splatted, not restated: these keys are the cache key cell-06 compares
    # against before reusing a .npy, so the two must not drift apart.
    **CACHE_KEY,
    "sequence_scope": pipeline.config.sequence_scope,
    "sequence_key": list(GROUP_COLS),
    "tokens_per_transaction": TOKENS_PER_TXN,
    "max_length": MAX_LENGTH,
    "embedding_dim": int(inference.embedding_dim),
    "splits": {k: int(v) for k, v in split_sizes.items()},
    "feature_columns": FEATURE_COLS,
    "datetime_column": SCHEMA.datetime,
    "label_column": "_target",
    # Train rows whose customers fitted the attention head; NB05 leaves them out.
    "head_rows_file": "train_head_rows.npy" if "learned" in POOLING else None,
}
(EMBED_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))

print("Embedding extraction complete.")
for split, n in split_sizes.items():
    labels = np.load(EMBED_DIR / f"{split}_labels.npy")
    print(f"  {split:<6}: {n:>9,} rows, {int(labels.sum()):>7,} positives "
          f"({labels.mean():.4%})")
print(f"\nMetadata -> {EMBED_DIR / 'metadata.json'}")

## 3. Sanity Check: What Went Into the Model

Decoding one window back to token strings is the cheapest way to confirm the input is
what you think it is: a chronological run of transactions from a single account, ending
with the scored one, and no padding in the middle.

In [ ]:
probe_split = "val"
probe_full, probe_subset = SPLIT_SOURCES[probe_split]

probe = cudf.read_parquet(str(SPLIT_DIR / probe_full)).head(200_000)
probe_proc = pipeline.preprocess(probe)
probe_tokens = pipeline.transform(probe_proc)
probe_encoder = HistoryWindowEncoder.from_token_frame(
    probe_tokens, probe_proc, GROUP_COLS, tokenizer.vocab,
    unk_id=tokenizer.unk_token_id, window=HISTORY_WINDOW, max_length=MAX_LENGTH,
)

positions = probe_proc.groupby(GROUP_COLS).cumcount().to_pandas().values
deep_rows = np.where(positions >= min(HISTORY_WINDOW - 1, positions.max()))[0]
row = int(deep_rows[0]) if len(deep_rows) else 0

window_ids = probe_encoder.batch(row, row + 1)[0]
n_real = int((window_ids != tokenizer.pad_token_id).sum())
decoded = tokenizer.decode(window_ids).split()

print(f"Row {row}: sequence position {positions[row]}, {n_real} non-pad tokens "
      f"of {MAX_LENGTH}")
print(f"Transactions in window: {decoded.count('<sep>') + 1}")
print(f"Last non-pad token is <eos>: {decoded[-1] == '<eos>'}")

accth = sorted({t for t in decoded if t.startswith("ACCTH_")})
if accth:
    print(f"Distinct accounts in this window: {len(accth)} -> {accth[:6]}")
    print("  (customer-scoped sequences interleave a customer's accounts; the ACCTH "
          "token is how the model tells them apart)")
print("\nFirst two transactions in the window:")
print("  " + " ".join(decoded[: 2 * (TOKENS_PER_TXN + 1) + 1]))
print("\nScored (last) transaction:")
print("  " + " ".join(decoded[-(TOKENS_PER_TXN + 1):]))

## 4. Visualization

Project the 512d embeddings to 2D/3D. The 2D view is a static scatter (positive vs
normal); the 3D view is interactive with a dropdown to recolor by feature. Structure
that lines up with countries, movement codes or amount bands means the model learned the
field semantics; positives clustering apart means the embedding carries signal the
downstream model can use.

The projector is picked by what is installed, in order: **cuML UMAP** (GPU) ->
**umap-learn** (CPU) -> **scikit-learn PCA**. cuML is *not* required anywhere in this
blueprint; this is the only place that would use it, and only for diagnostics. PCA is
linear, so it separates less than UMAP -- fine for a sanity check, but do not read the
absence of clusters in a PCA plot as the absence of signal.

In [ ]:
def make_projector(n_components):
    """Return (name, axis_label, fit_transform) for the best projector available."""
    try:
        from cuml.manifold import UMAP as cumlUMAP
        import cupy as cp
        model = cumlUMAP(n_neighbors=15, n_components=n_components,
                         min_dist=0.1, random_state=42)
        return ("cuML UMAP (GPU)", "UMAP",
                lambda X: cp.asnumpy(model.fit_transform(cp.asarray(X))))
    except ImportError:
        pass
    try:
        from umap import UMAP
        model = UMAP(n_neighbors=15, n_components=n_components,
                     min_dist=0.1, random_state=42)
        return ("umap-learn (CPU)", "UMAP",
                lambda X: np.asarray(model.fit_transform(X)))
    except ImportError:
        pass
    from sklearn.decomposition import PCA
    model = PCA(n_components=n_components, random_state=42)
    return ("scikit-learn PCA (linear fallback)", "PC",
            lambda X: model.fit_transform(X))


val_emb = np.load(EMBED_DIR / "val_embeddings.npy")
val_lbl = np.load(EMBED_DIR / "val_labels.npy")
test_emb = np.load(EMBED_DIR / "test_embeddings.npy")
test_lbl = np.load(EMBED_DIR / "test_labels.npy")

embeddings = np.concatenate([val_emb, test_emb])
labels = np.concatenate([val_lbl, test_lbl])
raw_df = pd.concat(
    [
        cudf.read_parquet(str(EMBED_DIR / "val_features.parquet")).to_pandas(),
        cudf.read_parquet(str(EMBED_DIR / "test_features.parquet")).to_pandas(),
    ],
    ignore_index=True,
)
assert len(raw_df) == len(embeddings)

VIZ_SIZE = 50_000
np.random.seed(42)
if len(embeddings) > VIZ_SIZE:
    idx = np.random.choice(len(embeddings), VIZ_SIZE, replace=False)
else:
    idx = np.arange(len(embeddings))

subset_embeds = embeddings[idx]
subset_labels = labels[idx]
subset_raw = raw_df.iloc[idx].reset_index(drop=True)

projector_2d, axis_2d, project_2d = make_projector(2)
print(f"Projecting {len(subset_embeds):,} samples to 2D with {projector_2d}...")
umap_2d = project_2d(subset_embeds)
print(f"Projection complete: {umap_2d.shape}")

In [ ]:
plt.figure(figsize=(12, 8))
mask_pos = subset_labels == 1
plt.scatter(
    umap_2d[~mask_pos, 0], umap_2d[~mask_pos, 1],
    c="blue", alpha=0.08, s=0.7, label="Normal",
)
plt.scatter(
    umap_2d[mask_pos, 0], umap_2d[mask_pos, 1],
    c="red", alpha=0.6, s=10, label="Positive", edgecolor="k", linewidth=0.1,
)
plt.legend(loc="upper right", markerscale=2)
plt.title(
    f"Transaction embeddings ({projector_2d}, n={len(subset_embeds):,}, "
    f"history={HISTORY_WINDOW} txns)"
)
plt.xlabel(f"{axis_2d} 1")
plt.ylabel(f"{axis_2d} 2")
plt.tight_layout()
plt.savefig(EMBED_DIR / "umap_visualization.png", dpi=150)
plt.show()
print(f"Saved -> {EMBED_DIR / 'umap_visualization.png'}")

In [ ]:
import matplotlib
import plotly.graph_objects as go
from IPython.display import HTML, IFrame, display as ipy_display

amount = subset_raw[SCHEMA.amount].astype(float)
viz_df = pd.DataFrame(
    {
        "label": np.where(subset_labels == 1, "Positive", "Normal"),
        "country": subset_raw[SCHEMA.counterparty_country].astype(str).fillna("NA"),
        "dc_flag": subset_raw[SCHEMA.dc_flag].astype(str).fillna("NA"),
        "movement_code": subset_raw[SCHEMA.movement_code].astype(str),
        "status": subset_raw[SCHEMA.operations_status].astype(str),
        "hour": subset_raw[SCHEMA.hour].astype(str),
        "amount_band": pd.qcut(
            amount.rank(method="first"), q=8,
            labels=[f"q{i + 1}" for i in range(8)],
        ).astype(str),
    }
)

projector_3d, axis_3d, project_3d = make_projector(3)
print(f"Projecting {len(subset_embeds):,} samples to 3D with {projector_3d}...")
umap_3d = project_3d(subset_embeds)

TOGGLES = [
    ("Label", "label", None),
    ("Country (top 8)", "country", 8),
    ("Debit/Credit", "dc_flag", None),
    ("Movement code (top 10)", "movement_code", 10),
    ("Operations status", "status", None),
    ("Amount band (octiles)", "amount_band", None),
    ("Hour of day", "hour", None),
]
GRAY = "#d3d3d3"


def tab20_hex(n):
    cmap = matplotlib.colormaps["tab20"].resampled(max(n, 2))
    return [
        "#{:02x}{:02x}{:02x}".format(int(r * 255), int(g * 255), int(b * 255))
        for r, g, b, _ in [cmap(i) for i in range(n)]
    ]


color_arrays = []
for _label, col, top_n in TOGGLES:
    values = viz_df[col].values
    if top_n is not None:
        keys = viz_df[col].value_counts().head(top_n).index.tolist()
    else:
        keys = sorted(viz_df[col].dropna().unique())
    color_map = dict(zip(keys, tab20_hex(len(keys))))
    color_arrays.append([color_map.get(v, GRAY) for v in values])

hover = [
    f"Label: {r.label}<br>Country: {r.country}<br>DC: {r.dc_flag}<br>"
    f"Movement: {r.movement_code}<br>Status: {r.status}<br>Amount: {r.amount_band}"
    for r in viz_df.itertuples()
]

fig3d = go.Figure(
    go.Scatter3d(
        x=umap_3d[:, 0], y=umap_3d[:, 1], z=umap_3d[:, 2],
        mode="markers",
        marker=dict(size=1.5, color=color_arrays[0], opacity=0.5),
        text=hover,
        hoverinfo="text",
        name=TOGGLES[0][0],
    )
)
fig3d.update_layout(
    title=dict(text=f"3D embedding explorer -- {TOGGLES[0][0]} (n={len(viz_df):,})",
               x=0.5, xanchor="center"),
    scene=dict(xaxis_title=f"{axis_3d} 1", yaxis_title=f"{axis_3d} 2",
               zaxis_title=f"{axis_3d} 3"),
    updatemenus=[
        dict(
            type="dropdown", direction="down", x=0.01, xanchor="left", y=0.97,
            yanchor="top", bgcolor="white", bordercolor="gray",
            buttons=[
                dict(
                    label=label,
                    method="update",
                    args=[
                        {"marker.color": [color_arrays[i]], "name": [label]},
                        {"title.text": f"3D embedding explorer -- {label}"},
                    ],
                )
                for i, (label, _col, _top) in enumerate(TOGGLES)
            ],
        )
    ],
    width=950, height=750, margin=dict(l=0, r=0, b=0, t=80), showlegend=False,
)

html_path = EMBED_DIR / "umap_3d_interactive.html"
fig3d.write_html(str(html_path), include_plotlyjs=True)
print(f"Saved -> {html_path}")

try:
    ipy_display(IFrame(src=str(html_path.relative_to(PROJECT_ROOT)), width="100%", height="780px"))
except Exception:
    ipy_display(HTML(f'<b>Interactive 3D plot saved:</b> <a href="{html_path}">{html_path.name}</a>'))

## Summary

Each labelled transaction now has a 512d embedding computed from its account's trailing
history, with labels and raw features saved in the same row order.

**Outputs**

| File | Contents |
|---|---|
| `embeddings/{split}_embeddings.npy` | (n, 512) last-token embeddings |
| `embeddings/{split}_labels.npy` | (n,) binary labels, same order |
| `embeddings/{split}_features.parquet` | 16 raw features + `_target` + `__row_id__`, same order |
| `embeddings/metadata.json` | window size, model path, split sizes |
| `embeddings/umap_visualization.png`, `umap_3d_interactive.html` | diagnostics |

Continue to [05_xgboost_fraud_detection.ipynb](./05_xgboost_fraud_detection.ipynb).